# Step 1 – Data Collection for Line Following

Drive the robot manually along the track while capturing labelled images.
Each frame is saved under one of three class folders:
- `dataset/forward/`
- `dataset/left/`
- `dataset/right/`

**How to use:**
1. Run **Cell 1** to start the camera — a live preview appears immediately.
2. Run **Cell 2**, then **click the grey box** that appears to give it keyboard focus.
3. **Type w/a/s/d** into the text box — the robot moves continuously and saves images every 0.3 s.
4. **Type any other key** (e.g. space) to stop the robot.
5. `s` moves backward for repositioning but frames are **not** saved.
6. Run **Cell 3** when done. Aim for ~200+ images per class.

In [1]:
# ── Cell 1: Imports and camera setup ─────────────────────────────────────────
import os, cv2, time, threading
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import pyzed.sl as sl
import motors
from traitlets.config.configurable import SingletonConfigurable
import traitlets

# Create dataset directories
for cls in ['forward', 'left', 'right']:
    os.makedirs(f'dataset/{cls}', exist_ok=True)

robot = motors.MotorsYukon(mecanum=False)

def bgr8_to_jpeg(img):
    return bytes(cv2.imencode('.jpg', img)[1])

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super().__init__()
        self.zed = sl.Camera()
        init = sl.InitParameters()
        init.camera_resolution = sl.RESOLUTION.VGA
        init.depth_mode = sl.DEPTH_MODE.PERFORMANCE
        init.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera open failed:', status)
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        info = self.zed.get_camera_information()
        self.width  = info.camera_configuration.resolution.width
        self.height = info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                raw = self.image.get_data()
                self.color_value = cv2.cvtColor(raw, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

# Clear any leftover singleton from a previous run
if Camera._instance is not None:
    try:
        Camera._instance.stop()
        Camera._instance.zed.close()
    except Exception:
        pass
    Camera.clear_instance()

camera = Camera()
camera.start()

# Live preview — shows what the robot sees as soon as the camera starts
preview_widget = widgets.Image(format='jpeg', width='50%')
preview_label  = widgets.Label(value='Live camera feed — bottom half (red line) is what gets saved')
display(widgets.VBox([preview_widget, preview_label]))

def preview_loop():
    while camera.thread_runnning_flag:
        if camera.color_value is not None:
            frame = camera.color_value.copy()
            h = frame.shape[0]
            # Draw a red line showing the crop boundary used when saving images
            cv2.line(frame, (0, h // 3), (frame.shape[1], h // 3), (0, 0, 255), 2)
            cv2.putText(frame, 'saved region below', (5, h // 3 - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
            disp = cv2.resize(frame, None, fx=0.5, fy=0.5)
            preview_widget.value = bgr8_to_jpeg(disp)
        time.sleep(0.05)

preview_thread = threading.Thread(target=preview_loop, daemon=True)
preview_thread.start()
print('Camera started. Live feed active above. Proceed to Cell 2 to start collecting data.')

[2026-03-19 12:44:21 UTC][ZED][INFO] Logging level INFO
[2026-03-19 12:44:21 UTC][ZED][INFO] Logging level INFO
[2026-03-19 12:44:21 UTC][ZED][INFO] Logging level INFO
[2026-03-19 12:44:22 UTC][ZED][INFO] [Init]  Depth mode: PERFORMANCE
[2026-03-19 12:44:23 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-03-19 12:44:23 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-03-19 12:44:23 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-03-19 12:44:23 UTC][ZED][INFO] [Init]  Serial Number: S/N 30505807
[2026-03-19 12:44:23 UTC][ZED][WARNING] [Init]  Self-calibration failed. Point the camera towards a more textured and brighter area. Avoid objects closer than 1 meter (Error code: 0x01) 


Camera started. Live feed active above. Proceed to Cell 2 to start collecting data.


In [2]:
# ── Cell 2: Continuous movement + labelled data capture ──────────────────────
# Type w/a/s/d into the text box to drive. The robot moves continuously
# until you type any other key or clear the box.
# Images are saved automatically every 0.3 s while a direction is active.
# 's' moves backward for repositioning but does NOT save frames.

DRIVE_SPEED   = 0.35
SAVE_INTERVAL = 0.2  # save more frequently to capture more turning data
CROP_START    = 3    # crop from 1/3 down — captures more of the track ahead

counts = {'forward': 0, 'left': 0, 'right': 0}
current_action = {'label': None}

# ── Widgets ───────────────────────────────────────────────────────────────────
display_widget = widgets.Image(format='jpeg', width='50%')
count_label    = widgets.Label(value='Saved — forward:0  left:0  right:0')
status_label   = widgets.Label(value='Status: idle')
text_input     = widgets.Text(value='', placeholder='type w/a/s/d', description='Drive:', layout=widgets.Layout(width='200px'))
display(widgets.VBox([display_widget, count_label, status_label, text_input]))

# ── Helpers ───────────────────────────────────────────────────────────────────
def update_count_label():
    total = sum(counts.values())
    warn  = ''
    if total > 30:
        mx = max(counts.values())
        mn = min(counts.values())
        if mn > 0 and mx / mn > 2.5:
            low = [k for k, v in counts.items() if v == mn][0]
            warn = f'  ⚠ collect more {low}!'
    count_label.value = f"Saved — forward:{counts['forward']}  left:{counts['left']}  right:{counts['right']}{warn}"

def save_frame(label):
    if camera.color_value is None:
        return
    frame = camera.color_value.copy()
    h = frame.shape[0]
    crop = cv2.resize(frame[h//CROP_START:, :], (224, 224))
    cv2.imwrite(f'dataset/{label}/{label}_{counts[label]:05d}.jpg', crop)
    counts[label] += 1
    update_count_label()

# ── Continuous drive + auto-save loop ────────────────────────────────────────
def drive_loop():
    last_save = 0.0
    while camera.thread_runnning_flag:
        label = current_action['label']
        now   = time.time()
        if label is not None and (now - last_save) >= SAVE_INTERVAL:
            save_frame(label)
            last_save = now
        time.sleep(0.05)

drive_thread = threading.Thread(target=drive_loop, daemon=True)
drive_thread.start()

# ── Text box observer ─────────────────────────────────────────────────────────
KEY_MAP = {
    'w': ('forward', lambda: robot.forward(DRIVE_SPEED)),
    'a': ('left',    lambda: robot.left(DRIVE_SPEED)),
    'd': ('right',   lambda: robot.right(DRIVE_SPEED)),
    's': (None,      lambda: robot.backward(DRIVE_SPEED)),
}

def on_text_change(change):
    val = change['new']
    if not val:
        return
    key = val[-1]   # only look at the most recently typed character
    if key in KEY_MAP:
        label, action = KEY_MAP[key]
        current_action['label'] = label
        action()
        desc = label.upper() if label else 'BACKWARD (not saved)'
        status_label.value = f'Status: {desc}'
    else:
        current_action['label'] = None
        robot.stop()
        status_label.value = 'Status: idle'

text_input.observe(on_text_change, names='value')

# ── Live display loop ─────────────────────────────────────────────────────────
def display_loop():
    while camera.thread_runnning_flag:
        if camera.color_value is not None:
            frame = camera.color_value.copy()
            h, w  = frame.shape[:2]
            cv2.line(frame, (0, h//3), (w, h//3), (0, 0, 255), 2)
            disp  = cv2.resize(frame, None, fx=0.5, fy=0.5)
            label = current_action['label']
            overlay = label.upper() if label else 'IDLE'
            cv2.putText(disp, f"{overlay}  F:{counts['forward']} L:{counts['left']} R:{counts['right']}",
                        (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            display_widget.value = bgr8_to_jpeg(disp)
        time.sleep(0.05)

disp_thread = threading.Thread(target=display_loop, daemon=True)
disp_thread.start()

print('Ready. Click the text box and type w/a/s/d to drive. Type any other key to stop.')

Ready. Click the text box and type w/a/s/d to drive. Type any other key to stop.


In [3]:
# ── Cell 3: Stop camera ───────────────────────────────────────────────────────
robot.stop()
try:
    camera.stop()
except Exception:
    pass

for cls in ['forward', 'left', 'right']:
    os.makedirs(f'dataset/{cls}', exist_ok=True)
    n = len(os.listdir(f'dataset/{cls}'))
    print(f'  {cls}: {n} images')
print('Done. Proceed to Step2_Train_CNN.ipynb')

  forward: 629 images
  left: 148 images
  right: 144 images
Done. Proceed to Step2_Train_CNN.ipynb
